# 🤖 Smart Q&A Bot — Structured Output with LangChain

## Learning Objectives
In this notebook, you will learn:
1. **Structured Output** - How to constrain an LLM's response to a validated Pydantic schema using `with_structured_output`
2. **Production Bot Design** - How to wrap a prompt + LLM chain inside a reusable class with graceful error handling
3. **Batch Processing** - How to answer multiple questions concurrently with `chain.batch()`
4. **Observability** - How to trace chain executions with LangSmith's `@traceable` decorator

## Prerequisites
- Completion of the earlier LangChain Foundations notebooks (prompts, chains, LCEL)
- An `OPENAI_API_KEY` in your `.env` file
- (Optional) A `LANGSMITH_API_KEY` in your `.env` file to enable tracing

---
## 🔧 Part 1: Environment Setup

We load environment variables from `.env`, import the LangChain, Pydantic, and LangSmith building blocks, and enable LangSmith tracing when an API key is present so every chain call in this notebook shows up in your LangSmith project.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, dotenv, and LangSmith Tracing
# ============================================================================
import os
from typing import List

from dotenv import load_dotenv
from langsmith import Client, traceable
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

# -- LangSmith Configuration --
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "Smart Q&A Bot Project")
    print(f"LangSmith is configured. - Project: {os.getenv('LANGSMITH_PROJECT')}")

print("✅ Environment setup complete!")


---
## 📐 Part 2: Defining the Structured Output Schema

Before calling the LLM, we define a Pydantic schema describing exactly what a valid answer should look like. LangChain's `with_structured_output` uses this schema to force the model to return validated, typed data instead of free-form text.

### Key Concepts:
- **Structured Output**: Constraining an LLM's response to a schema instead of raw text
- **Pydantic `Field`**: Adds descriptions that guide the LLM's understanding of each field's purpose

### 📦 `QAResponse` Model

The schema returned by every call to the bot — it captures the answer, a confidence level, the reasoning behind it, suggested follow-up questions, and whether external sources are needed.

In [ ]:
# ============================================================================
# QARESPONSE: Structured Output Schema
# ============================================================================
class QAResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    confidence: str = Field(description="Confidence level: high, medium, or low")
    reasoning: str = Field(description="The reasoning behind the answer provided.")
    follow_up_questions: List[str] = Field(
        description="A list of follow-up questions related to the topic.",
        default_factory=list,
    )
    sources_needed: bool = Field(
        description="Indicates whether sources are needed for the answer.",
        default=False,
    )


---
## 🧠 Part 3: Building the `SmartQABot` Class

The core bot class wraps a prompt template and an OpenAI chat model bound to the `QAResponse` schema via `with_structured_output`. It exposes `ask()` for single questions — with graceful error handling that returns a valid `QAResponse` even on failure — and `ask_batch()` for concurrent batch processing. Both methods are wrapped in LangSmith `@traceable` decorators for observability.

In [ ]:
# ============================================================================
# SMARTQABOT: Production Q&A Bot Class
# ============================================================================
class SmartQABot:
    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        temperature: float = 0.3,
    ):
        self.model = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        ).with_structured_output(QAResponse)
        self.prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """You are a knowledgeable Q&A assistant.

Your guidelines:
- Answer questions accurately and concisely
- Be honest about uncertainty - set confidence to 'low' if unsure
- Provide clear reasoning for your answers
- Suggest relevant follow-up questions
- Indicate if external sources would help

Always respond with accurate, helpful information.""",
                ),
                ("human", "{question}"),
            ]
        )
        self.chain = self.prompt | self.model

    @traceable(name="ask_question", run_type="chain")
    def ask(self, question: str) -> QAResponse:
        try:
            response = self.chain.invoke({"question": question})
            return response
        except Exception as e:
            # return a greaceful error response
            return QAResponse(
                answer="I'm sorry, I couldn't process your question at this time.",
                confidence="low",
                reasoning=str(e),
                follow_up_questions=["Could you please try again later?"],
                sources_needed=True,
            )

    @traceable(name="ask_batch", run_type="chain")
    def ask_batch(self, questions: List[str]) -> List[QAResponse]:
        """Ask multiple questions in parallel."""
        inputs = [{"question": q} for q in questions]
        return self.chain.batch(inputs)


---
## 🎬 Part 4: Demo Functions

With the schema and bot class in place, these three demo functions exercise the bot end to end: a basic multi-question run, an edge-case error-handling run, and a concurrent batch run. Each is wrapped in a LangSmith `@traceable` span so you can inspect the calls in your LangSmith project.

### 4.1 🗣️ Basic Q&A Demo

`demo_qa_bot` instantiates the bot and asks it three unrelated questions one at a time, printing the full structured response — answer, confidence, reasoning, follow-ups, and whether sources are needed — for each.

In [ ]:
# ============================================================================
# DEMO_QA_BOT: Basic Q&A Demonstration
# ============================================================================
# Demo Usage
def demo_qa_bot():
    bot = SmartQABot()

    questions = [
        "What is the capital of France?",
        "Explain the theory of relativity.",
        "How does photosynthesis work?",
    ]

    print("=" * 60)
    print("SMART Q&A BOT DEMO")
    print("=" * 60)

    for question in questions:

        print(f"\n Question: {question}")
        print("-" * 40)

        response = bot.ask(question)

        print(f"Question: {question}")
        print(f"Answer: {response.answer}")
        print(f"Confidence: {response.confidence}")
        print(f"Reasoning: {response.reasoning}")
        print(f"Follow-up Questions: {response.follow_up_questions}")
        print(f"Sources Needed: {response.sources_needed}")
        print("-" * 60)


### 4.2 ⚠️ Error Handling Demo

`demo_error_handling` demonstrates the bot's graceful failure path by sending an artificially long, repetitive question and confirming that `ask()` still returns a usable `QAResponse` instead of raising an unhandled exception.

In [ ]:
# ============================================================================
# DEMO_ERROR_HANDLING: Graceful Failure Demonstration
# ============================================================================
@traceable(name="error_handling_demo", run_type="chain")
def demo_error_handling():
    """Demonstrate error handling."""

    bot = SmartQABot()

    print("\n" + "=" * 60)
    print("ERROR HANDLING DEMO")
    print("=" * 60)

    # Test with a very long question (edge case)
    long_question = "What is " + "very " * 100 + "important?"

    response = bot.ask(long_question)
    print(f"Handled gracefully: {response.confidence}")


### 4.3 📦 Batch Processing Demo

`demo_batch_processing` shows `ask_batch()` answering several questions concurrently via `chain.batch()`, then prints a truncated answer and confidence level for each.

In [ ]:
# ============================================================================
# DEMO_BATCH_PROCESSING: Concurrent Batch Demonstration
# ============================================================================
@traceable(name="batch_demo", run_type="chain")
def demo_batch_processing():
    """Demonstrate batch processing."""

    bot = SmartQABot()

    questions = [
        "What is Python?",
        "What is JavaScript?",
        "What is Rust?",
    ]

    print("\n" + "=" * 60)
    print("BATCH PROCESSING DEMO")
    print("=" * 60)

    responses = bot.ask_batch(questions)

    for q, r in zip(questions, responses):
        print(f"\n{q}")
        print(f"  -> {r.answer[:100]}...")
        print(f"  Confidence: {r.confidence}")


---
## ▶️ Part 5: Running the Demos

The original `__main__` guard is kept verbatim below. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is when executed — it runs all three demos in sequence and prints a closing summary of what the section covered.

In [ ]:
# ============================================================================
# RUN: Execute All Demos
# ============================================================================
if __name__ == "__main__":

    try:
        demo_qa_bot()
        demo_batch_processing()
        demo_error_handling()

        print("\n" + "=" * 60)
        print("Section 1 Complete!")
        print("=" * 60)
        print(
            """
What you learned:
- LangChain ecosystem overview
- Environment setup with uv
- Core concepts: Runnables, LCEL, pipe operator
- Working with multiple LLM providers
- Prompt templates and message types
- Output parsers and structured output
- Building a production Q&A bot
- LangSmith tracing with @traceable decorator

Next: Section 2 - Chains, RAG & Memory
        """
        )
    finally:
        pass


---
## 📝 Summary

In this notebook, we built a production-style Q&A bot on top of LangChain's structured-output and tracing features.

### 1. Schema & Bot Design
- **`QAResponse`**: A Pydantic schema that forces the LLM to return a typed answer, confidence level, reasoning, follow-up questions, and a sources-needed flag
- **`SmartQABot`**: Wraps a prompt + `ChatOpenAI` model (via `with_structured_output`) into a reusable class with `ask()` and `ask_batch()`
- **Graceful error handling**: `ask()` catches exceptions and still returns a valid `QAResponse` instead of crashing

### 2. Observability & Batching
- **`@traceable`**: Every bot method is wrapped so calls are visible in your LangSmith project when `LANGSMITH_API_KEY` is set
- **`chain.batch()`**: Answers multiple questions concurrently instead of looping one at a time

### Next Steps
- This is the final notebook in `01_LangChain_Foundations/`
- Continue to **`02_RAG_and_Retrieval/`** to learn how to ground bots like this one in your own documents with retrieval-augmented generation